In [4]:
import cv2
import numpy as np
import tensorflow as tf

# 1. Load models
mask_model = tf.keras.models.load_model('1st_try.keras')
class_names = ['with_mask', 'without_mask'] 
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# 2. Start webcam
cap = cv2.VideoCapture(0)
print("Webcam is active. Click on the video window and press 'q' to quit.")

while True:
    ret, frame = cap.read()
    if not ret:
        break
        
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray_frame, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))
    
    for (x, y, w, h) in faces:
        face_roi = frame[y:y+h, x:x+w]
        face_resized = cv2.resize(face_roi, (200, 200))
        face_rgb = cv2.cvtColor(face_resized, cv2.COLOR_BGR2RGB)
        
        face_array = tf.keras.utils.img_to_array(face_rgb) / 255.0
        face_array = np.expand_dims(face_array, axis=0)
        
        predictions = mask_model.predict(face_array, verbose=0)
        predicted_class_idx = np.argmax(predictions, axis=-1)[0]
        confidence = np.max(predictions)
        label = class_names[predicted_class_idx]
        
        center_x = x + w // 2
        center_y = y + h // 2
        radius = int((w + h) / 4)
        
        color = (0, 255, 0) if label == 'with_mask' else (0, 0, 255)
        cv2.circle(frame, (center_x, center_y), radius, color, thickness=3)
        cv2.putText(frame, f"{label}: {confidence*100:.1f}%", (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    
    # Open a dedicated window to show the video
    cv2.imshow('Live Mask Detection', frame)
    
    # ---------------------------------------------------------
    # THE 'Q' QUIT LOGIC
    # ---------------------------------------------------------
    # waitKey(1) pauses the script for 1 millisecond to listen for a key press.
    # If the key pressed is 'q', the loop breaks.
    if cv2.waitKey(1) & 0xFF == ord('q'):
        print("Quit signal received. Closing...")
        break

# 3. Clean up and force window close (Mac Freeze Fix)
cap.release()
cv2.destroyAllWindows()

# On Macs running Jupyter, the window manager sometimes refuses to destroy 
# the window immediately. Calling waitKey a few times forces it to flush and close.
for i in range(10):
    cv2.waitKey(1)

Webcam is active. Click on the video window and press 'q' to quit.
Quit signal received. Closing...
